In [7]:
import sys
import dotenv
import textwrap
import langextract as lx
import os

In [11]:
dotenv.load_dotenv('.env.local5')

True

In [12]:
# 1. Define the prompt and extraction rules
prompt = textwrap.dedent("""\
    Extract characters, emotions, and relationships in order of appearance.
    Use exact text for extractions. Do not paraphrase or overlap entities.
    Provide meaningful attributes for each entity to add context.""")

# 2. Provide a high-quality example to guide the model
examples = [
    lx.data.ExampleData(
        text="ROMEO. But soft! What light through yonder window breaks? It is the east, and Juliet is the sun.",
        extractions=[
            lx.data.Extraction(
                extraction_class="character",
                extraction_text="ROMEO",
                attributes={"emotional_state": "wonder"}
            ),
            lx.data.Extraction(
                extraction_class="emotion",
                extraction_text="But soft!",
                attributes={"feeling": "gentle awe"}
            ),
            lx.data.Extraction(
                extraction_class="relationship",
                extraction_text="Juliet is the sun",
                attributes={"type": "metaphor"}
            ),
        ]
    )
]

In [13]:
# The input text to be processed
input_text = "Lady Juliet gazed longingly at the stars, her heart aching for Romeo"

# Run the extraction
result = lx.extract(
    text_or_documents=input_text,
    prompt_description=prompt,
    examples=examples,
    model_id=f"azureopenai-{os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')}",
)

In [15]:
result.extractions

[Extraction(extraction_class='character', extraction_text='Lady Juliet', char_interval=CharInterval(start_pos=0, end_pos=11), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=1, group_index=0, description=None, attributes={'title': 'Lady', 'emotional_state': 'gazing with longing'}),
 Extraction(extraction_class='emotion', extraction_text='longingly', char_interval=CharInterval(start_pos=18, end_pos=27), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=2, group_index=1, description=None, attributes={'feeling': 'longing'}),
 Extraction(extraction_class='emotion', extraction_text='her heart aching', char_interval=CharInterval(start_pos=42, end_pos=58), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=3, group_index=2, description=None, attributes={'intensity': 'strong', 'subject': 'Juliet'}),
 Extraction(extraction_class='relationship', extraction_text='for Romeo', char_interval=CharInterval(start

In [46]:
# Test both simple extraction and classification to ensure attributes work consistently
print("=== SIMPLE EXTRACTION TEST ===")

# Simple character extraction example
simple_prompt = textwrap.dedent("""Extract all characters mentioned in this text.""")

simple_examples = [
    lx.data.ExampleData(
        text="Romeo loved Juliet deeply.",
        extractions=[
            lx.data.Extraction(
                extraction_class="Character",
                extraction_text="Romeo",
                attributes={"role": "protagonist", "emotion": "love"}
            ),
            lx.data.Extraction(
                extraction_class="Character", 
                extraction_text="Juliet",
                attributes={"role": "protagonist", "emotion": "beloved"}
            ),
        ]
    )
]

simple_text = "In this tale, Romeo was passionate and Juliet was beautiful."

# Extract using simple prompt
simple_result = lx.extract(
    text_or_documents=simple_text,
    prompt_description=simple_prompt,
    examples=simple_examples,
    debug=False,
    model_id=f"azureopenai-{os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')}",
)

print(f"Simple Result: {simple_result}")
print("Simple Extractions:", simple_result.extractions)
for i, extraction in enumerate(simple_result.extractions):
    print(f"Simple Extraction {i}: {extraction}")
    print(f"Attributes: {extraction.attributes}")

print("\n=== CLASSIFICATION TEST ===")

# Use the existing classification test
classify_result = lx.extract(
    text_or_documents=text,
    prompt_description=classify_prompt,
    examples=classify_examples,
    debug=False,
    model_id=f"azureopenai-{os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')}",
)

print(f"Classification Result: {classify_result}")
print("Classification Extractions:", classify_result.extractions)
for i, extraction in enumerate(classify_result.extractions):
    print(f"Classification Extraction {i}: {extraction}")
    print(f"Attributes: {extraction.attributes}")

print("\n=== SUMMARY ===")
print("✓ Simple extraction working with attributes" if simple_result.extractions and all(e.attributes for e in simple_result.extractions) else "✗ Simple extraction failed")
print("✓ Classification working with attributes" if classify_result.extractions and all(e.attributes for e in classify_result.extractions) else "✗ Classification failed")
print("✓ Both examples working consistently!" if all([
    simple_result.extractions and all(e.attributes for e in simple_result.extractions),
    classify_result.extractions and all(e.attributes for e in classify_result.extractions)
]) else "✗ Some examples not working")

=== SIMPLE EXTRACTION TEST ===
Simple Result: AnnotatedDocument(extractions=[Extraction(extraction_class='Character', extraction_text='Romeo', char_interval=CharInterval(start_pos=14, end_pos=19), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=1, group_index=0, description=None, attributes={'role': 'protagonist', 'emotion': 'passionate'}), Extraction(extraction_class='Character', extraction_text='Juliet', char_interval=CharInterval(start_pos=39, end_pos=45), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=2, group_index=1, description=None, attributes={'role': 'protagonist', 'emotion': 'beautiful'})], text='In this tale, Romeo was passionate and Juliet was beautiful.')
Simple Extractions: [Extraction(extraction_class='Character', extraction_text='Romeo', char_interval=CharInterval(start_pos=14, end_pos=19), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=1, group_index=0, description=None, 